# STAR: Vehicle monitoring (licence plates)

Code for the vehicle-monitoring results of
**"STAR: Semantic-Criticality Tiered Allocation of Radio Resources via Zero-Shot Learning"**.
SCI estimation and the physical layer are the same as in the factory notebook; only the dataset and the
query change (the plate task).

| Section | Paper |
|---|---|
| 2. OpenALPR US dataset | Sec. IV, Sec. IV-C |
| 3. SCI estimation via zero-shot learning | Algorithm 1 |
| 4. Tiered transmission and reception | Sec. III-B |
| 5. Speaker + judge evaluation | Sec. IV-A |
| 6. Plate experiment | Fig. 8, Fig. 9 |
| 7. Received images | Fig. 7 |

**Dataset**: this notebook expects the US end-to-end subset of the
[OpenALPR benchmark](https://github.com/openalpr/benchmarks) in `vehicle_dataset/` (see the README).
Run from the **repository root**. Results go to `results/vehicle/` and figures to `results/figures/vehicle/`.

## 1. Setup

In [ ]:
import os, sys, gc, time, random, pickle
import numpy as np, torch, requests
import matplotlib.pyplot as plt
from PIL import Image as PILImage

REPO_ROOT   = os.path.abspath(".")          # run from the repository root
DATA_DIR    = os.path.join(REPO_ROOT, "vehicle_dataset")
RESULTS_DIR = os.path.join(REPO_ROOT, "results", "vehicle")
FIG_DIR     = os.path.join(REPO_ROOT, "results", "figures", "vehicle")
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
import torch, gc, time, requests, numpy as np
from PIL import Image as PILImage
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

HF_MODEL      = "Qwen/Qwen2.5-VL-7B-Instruct"
OLLAMA_URL    = "http://localhost:11434/api/generate"
SPEAKER_MODEL = "qwen3-vl:30b-a3b-instruct"     # vision speaker
JUDGE_MODEL   = "qwen3:8b"                        # text judge

def _gpu_used():
    if not torch.cuda.is_available(): return 0.0
    f,t = torch.cuda.mem_get_info(); return (t-f)/1e9

# ── HF (transformers) model ────────────────────────────────────────────────
def load_hf_model(force=False):
    """Load Qwen2.5-VL into globals model/processor (eager attn). Idempotent."""
    global model, processor
    if not force and globals().get("model") is not None and globals().get("processor") is not None:
        print(f"HF model already loaded | GPU {_gpu_used():.1f} GB"); return model, processor
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        HF_MODEL, dtype=torch.bfloat16, device_map="auto", attn_implementation="eager")
    processor = AutoProcessor.from_pretrained(HF_MODEL)
    print(f"loaded HF {HF_MODEL} | eager attn | GPU {_gpu_used():.1f} GB")
    return model, processor

def free_hf_model():
    """Drop the HF model/processor and clear VRAM. Safe if not loaded."""
    global model, processor
    for _n in ["model","processor"]:
        if globals().get(_n) is not None:
            globals()[_n] = None
            try: del globals()[_n]
            except Exception: pass
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.synchronize()
    print(f"freed HF model | GPU {_gpu_used():.1f} GB")

# ── Ollama models (served process; load/unload via keep_alive) ──────────────
def load_ollama(model_name, wait=2):
    """Warm up an Ollama model (a tiny request makes Ollama load it). keep_alive 30m."""
    try:
        requests.post(OLLAMA_URL, json={"model":model_name,"prompt":"ok","stream":False,
                                        "keep_alive":"30m","options":{"num_predict":1}}, timeout=120)
        time.sleep(wait)
        print(f"loaded Ollama: {model_name} | GPU {_gpu_used():.1f} GB")
    except Exception as ex:
        print(f"load {model_name} failed: {str(ex)[:50]}")

def unload_ollama(model_name, wait=5):
    """Ask Ollama to drop a model now (keep_alive 0). Async, so we pause."""
    try:
        requests.post(OLLAMA_URL, json={"model":model_name,"prompt":"","keep_alive":0}, timeout=60)
        time.sleep(wait)
        print(f"unloaded Ollama: {model_name} | GPU {_gpu_used():.1f} GB")
    except Exception as ex:
        print(f"unload {model_name} skipped: {str(ex)[:50]}")

# ── free everything (clean slate) ──────────────────────────────────────────
def free_all_models():
    """Free HF + both Ollama models. Call at the start of a run for a clean GPU."""
    free_hf_model()
    for m in [SPEAKER_MODEL, JUDGE_MODEL]:
        unload_ollama(m, wait=2)
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"── all models freed | GPU {_gpu_used():.1f} GB ──")

print("model manager ready: load_hf_model / free_hf_model / load_ollama / unload_ollama / free_all_models")

## 2. Licence-plate dataset (OpenALPR benchmark, US end-to-end)

Each `usXXX.jpg` in `vehicle_dataset/` has a matching `usXXX.txt` whose last tab-separated field is the plate.
Only landscape frames at least 1200 px wide are kept, and the 100 largest are used.

To use another dataset, change `DATA_DIR`, `load_records()` and `QUESTION_SPEC`; nothing downstream depends on it.

In [ ]:
import os, glob
from PIL import Image as PILImage

# ─── DATASET CONFIG ───────────────────────────────────────────────────────
N_PANELS   = 100          # how many images to keep
MIN_WIDTH  = 1200        # keep only wide, legible frames
LANDSCAPE_ONLY = True    # drop tall portrait shots (plate too small/far)

# question: text asked to the VLM | field: meta key with the ground truth | critical: protected (score 2)
QUESTION_SPEC = [
    {"question": "What is the license plate number of the vehicle?",
     "field": "plate", "critical": True},
]

# ─── Loader: read one image + ground-truth fields into a `meta` dict ───────
def _plate_from_txt(txt_path):
    line = open(txt_path).read().strip()
    if not line:
        return None
    return line.split("\t")[-1].strip().upper()   # OpenALPR: last tab field = plate

def load_records(data_dir):
    """Return list of {img, meta} for every valid image in data_dir.
       meta must contain every 'field' named in QUESTION_SPEC."""
    recs = []
    for img_path in sorted(glob.glob(os.path.join(data_dir, "*.jpg"))
                           + glob.glob(os.path.join(data_dir, "*.png"))):
        txt_path = os.path.splitext(img_path)[0] + ".txt"
        if not os.path.exists(txt_path):
            continue
        plate = _plate_from_txt(txt_path)
        if not plate:
            continue
        img = PILImage.open(img_path).convert("RGB")
        recs.append({
            "img": img,
            "meta": {
                "plate": plate,                 # ground-truth field(s)
                "path": img_path,
            },
        })
    return recs

# ─── Build panel_records (generic below this line; no per-dataset code) ────
def _keep(img):
    w, h = img.size
    if LANDSCAPE_ONLY and w <= h:
        return False
    return w >= MIN_WIDTH

_all = [r for r in load_records(DATA_DIR) if _keep(r["img"])]
_all = sorted(_all, key=lambda r: r["img"].size[0] * r["img"].size[1], reverse=True)

panel_records = []
for i, r in enumerate(_all[:N_PANELS]):
    panel_records.append({
        "panel_id": i,
        "img": r["img"],
        "meta": r["meta"],
    })

# ─── Question interface (unchanged names for the rest of the notebook) ─────
ACTIVE_QUESTION_TYPES = [s["question"] for s in QUESTION_SPEC]
QUESTION_TYPES = list(ACTIVE_QUESTION_TYPES)
CRITICAL_QUESTIONS = [s["question"] for s in QUESTION_SPEC if s["critical"]]
_Q_FIELD = {s["question"]: s["field"] for s in QUESTION_SPEC}

def answer_for_question(meta, q):
    return meta[_Q_FIELD[q]]

# ─── Summary ──────────────────────────────────────────────────────────────
print(f"loaded {len(panel_records)} panels from {DATA_DIR}")
print("questions:")
for s in QUESTION_SPEC:
    tag = "CRITICAL" if s["critical"] else "normal"
    print(f"  [{tag}] {s['question']}  <- meta['{s['field']}']")
print("\nsample panels:")
for r in panel_records[:5]:
    print(f"  P{r['panel_id']:2d}  plate={r['meta']['plate']:8s}  size={r['img'].size}")

from collections import Counter
sizes = Counter(r["img"].size for r in panel_records)
print("\nsmallest:", min(panel_records, key=lambda r: r['img'].size[0]*r['img'].size[1])["img"].size)
print("distinct sizes:", len(sizes))

In [ ]:
dataset = []
for p in panel_records:
    meta = p["meta"]
    qs   = list(ACTIVE_QUESTION_TYPES)
    ans  = [answer_for_question(meta, q) for q in qs]
    dataset.append({
        "panel_id":  p["panel_id"],
        "img":       p["img"],
        "questions": qs,
        "answers":   ans,
        "qa":        dict(zip(qs, ans)),
        "meta":      meta,
    })

# quick text summary
print(f"dataset: {len(dataset)} entries")
print("questions per entry:", len(dataset[0]["questions"]))
print("\nExample entry [0]:")
d0 = dataset[0]
print("  panel_id:", d0["panel_id"], "| size:", d0["img"].size)
for q, a in d0["qa"].items():
    print(f"    Q: {q}\n       A: {a}")


# show the first 3 entries with their ground-truth answers
picks = dataset[:3]
fig, axes = plt.subplots(1, 3, figsize=(22, 9))
for ax, d in zip(axes, picks):
    ax.imshow(d["img"])
    ax.axis("off")
    ax.set_title(f"panel {d['panel_id']}  ({d['img'].size[0]}×{d['img'].size[1]})",
                 fontsize=13, fontweight="bold")
    lines = [f"{q}  →  {a}" for q, a in d["qa"].items()]
    ax.text(0.5, -0.02, "\n".join(lines), transform=ax.transAxes,
            ha="center", va="top", fontsize=10, family="monospace")

plt.tight_layout()
plt.show()

# full detail for cross-check
for d in picks:
    print(f"\n=== panel {d['panel_id']} ===")
    for q, a in d["qa"].items():
        print(f"  {a!r:12}  ←  {q}")

## 3. SCI estimation via zero-shot learning (Algorithm 1)

A frozen **Qwen2.5-VL-7B** answers each task query with an `<answer>…</answer><where>…</where>` prompt.
The decoder attention of the `<where>` tokens onto the image tokens (mid-layer band 0.4L–0.7L, head mean,
max over tokens, attention-sink clipping, Gaussian blur, grey dilation) gives the per-task map $A_q$.
The maps are weighted by the task-criticality score $\omega_q$, max-combined, thresholded at $\tau=0.5$
and nearest-upsampled to the pixel grid, giving the transmit tensor $T=(R,G,B,S,\text{pos})$.

Eager attention is required (flash / SDPA do not return attention weights). Loading the 7B model in bfloat16
needs ~16 GB of GPU memory, so stop or unload Ollama first.

In [ ]:
load_hf_model()

In [ ]:
# Algorithm 1, lines 2-6: (image, question) → per-task map A_q on the Hm×Wm token grid
import torch, numpy as np, re
from scipy.ndimage import gaussian_filter, grey_dilation

ATTENTION_PROMPT = """Look at the image and answer using only what is visible.

Question: {question}

First give the exact answer value. Then describe WHERE in the image you see the
evidence for that answer — name the object or area and its physical location.


Format your response exactly as:
<answer>VALUE</answer>
<where>where you see the evidence</where>"""

def _find_span(pieces, open_tag, close_tag):
    start = end = None
    for i, p in enumerate(pieces):
        if open_tag in p and start is None: start = i + 1
        if close_tag in p and start is not None and end is None: end = i; break
    return start, end

def attention_heatmap(image, question, layer_frac=(0.4, 0.7), head_reduce="mean",
                      token_pool="max", sink_drop=4, smooth=1.2, dilate=3, verbose=False):
    prompt = ATTENTION_PROMPT.format(question=question)
    messages = [{"role":"user","content":[
        {"type":"image","image":image}, {"type":"text","text":prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    T, Hg, Wg = inputs["image_grid_thw"][0].tolist()
    merge = getattr(processor.image_processor, "merge_size", 2)
    Hm, Wm = Hg // merge, Wg // merge; n_vis = T * Hm * Wm
    ids = inputs["input_ids"][0].tolist()
    vs = processor.tokenizer.convert_tokens_to_ids("<|vision_start|>")
    ve = processor.tokenizer.convert_tokens_to_ids("<|vision_end|>")
    s = ids.index(vs) + 1; e = ids.index(ve); assert e - s == n_vis

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=48, do_sample=False,
                             output_attentions=True, return_dict_in_generate=True)
    gen_ids = out.sequences[0, inputs["input_ids"].shape[1]:]
    full = processor.batch_decode(gen_ids[None], skip_special_tokens=True)[0].strip()
    ans_m = re.search(r"<answer>(.*?)</answer>", full, re.S)
    whr_m = re.search(r"<where>(.*?)</where>", full, re.S)
    answer = (ans_m.group(1).strip() if ans_m else full.split("\n")[0])[:30]
    where  = (whr_m.group(1).strip() if whr_m else "")[:40]

    piece = [processor.tokenizer.decode([t]) for t in gen_ids.tolist()]
    ws, we_ = _find_span(piece, "<where>", "</where>")
    if ws is None or we_ is None or we_ <= ws:
        ws, we_ = _find_span(piece, "<answer>", "</answer>")
    if ws is None or we_ is None or we_ <= ws:
        ws, we_ = 0, len(piece)
    steps = [i for i in range(ws, we_) if i < len(out.attentions)]
    if not steps: steps = [len(out.attentions)-1]
    if verbose: print(f"  [{question[:30]}…] A={answer!r} WHERE={where!r} | {len(steps)} tok")

    maps = []
    for st in steps:
        step = out.attentions[st]; L = len(step)
        lo, hi = int(L*layer_frac[0]), max(int(L*layer_frac[1]), int(L*layer_frac[0])+1)
        sel = torch.stack(step[lo:hi]).mean(0)[0]
        a = sel.mean(0) if head_reduce=="mean" else sel.max(0).values
        maps.append(a[-1].float().cpu().numpy()[s:e])
    M = np.stack(maps, 0)
    img_attn = M.max(0) if token_pool=="max" else M.mean(0)

    if sink_drop > 0:
        thr = np.sort(img_attn)[-sink_drop-1]; img_attn = np.clip(img_attn, None, thr)
    grid = img_attn.reshape(Hm, Wm).astype(np.float32)
    raw_peak = float(grid.max() / (grid.mean() + 1e-8))
    if smooth and smooth > 0: grid = gaussian_filter(grid, sigma=smooth)
    if dilate and dilate > 1: grid = grey_dilation(grid, size=(dilate, dilate))
    heat = (grid - grid.min()) / (np.ptp(grid) + 1e-8)
    return answer, heat, raw_peak

print("attention_heatmap ready (Algorithm 1)")

Check: the attention of the plate task must land on the licence plate, not on the car body.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from PIL import Image as PILImage

N_SHOW = 8
TUNE = dict(smooth=1.2, dilate=3, sink_drop=4, token_pool="max")

# use the critical question(s) declared in the dataset cell
QLIST = CRITICAL_QUESTIONS if CRITICAL_QUESTIONS else ACTIVE_QUESTION_TYPES
q = QLIST[0]
print(f"question: {q!r}")

tests = [(p["img"], p["meta"].get("plate", "?"), p["panel_id"])
         for p in panel_records[:N_SHOW]]

n = len(tests); cols = 2; rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4.4*rows)); axes = np.array(axes).reshape(-1)
results = []
for i, (img, gt, pid) in enumerate(tests):
    ans, heat, peak = attention_heatmap(img, q, **TUNE)
    npimg = np.array(img.convert("RGB")); H, W = npimg.shape[:2]
    hu = np.array(PILImage.fromarray((heat*255).astype(np.uint8)).resize((W, H), PILImage.BILINEAR))/255.0
    ax = axes[i]; ax.imshow(npimg); ax.imshow(hu, cmap="jet", alpha=0.45); ax.axis("off")
    ax.set_title(f"panel {pid} | GT:{gt}\nVLM:{ans[:24]!r} | peak={peak:.1f}", fontsize=9)
    results.append((pid, gt, ans, peak))
for j in range(n, len(axes)): axes[j].axis("off")
plt.suptitle(f"Algorithm 1 — plate attention sweep  {TUNE}", fontsize=13, y=1.005)
plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

print("\nRanked by concentration:")
for pid, gt, a, p in sorted(results, key=lambda r: -r[3]):
    print(f"  peak={p:5.1f} | panel {pid:2d} | GT:{gt:8s} | VLM:{a[:30]!r}")

In [ ]:
# Algorithm 1, line 8: weight each A_q by its score and max-combine
import numpy as np

# Critical set is driven by the dataset cell's CRITICAL_QUESTIONS.
# qids index into ACTIVE_QUESTION_TYPES; each critical question gets score 2.
CRITICAL = {ACTIVE_QUESTION_TYPES.index(q): 2 for q in CRITICAL_QUESTIONS}

def importance_map(image, critical=CRITICAL, combine="max", **tune):
    """Run each critical question, weight by score, combine → one token-grid map.
       combine='max' (union) or 'sum' (shared-importance)."""
    per_q = {}
    stacked = []
    for qid, score in critical.items():
        q = ACTIVE_QUESTION_TYPES[qid]
        ans, heat, peak = attention_heatmap(image, q, **tune)   # heat 0..1, Hm×Wm
        per_q[qid] = (q, ans, heat, peak)
        stacked.append(score * heat)                            # weight by score
    S = np.stack(stacked, 0)
    combined = S.max(0) if combine == "max" else S.sum(0)
    combined = (combined - combined.min()) / (np.ptp(combined) + 1e-8)
    return combined, per_q

print("importance_map ready | critical qids:", list(CRITICAL))

In [ ]:
import random
SEED = random.randrange(2**32)
rng = random.Random(SEED)
print(f"[importance_map debug] SEED = {SEED}")

# critical set for this debug: use the dataset's critical questions
CRITICAL_DEBUG = dict(CRITICAL)     # {qid: score} from the importance_map cell
TUNE = dict(smooth=1.2, dilate=3, sink_drop=4, token_pool="max")

print("critical questions:")
for qid, sc in CRITICAL_DEBUG.items():
    print(f"  q{qid} (score {sc}): {ACTIVE_QUESTION_TYPES[qid]}")

def show_importance(image, title=""):
    combined, per_q = importance_map(image, CRITICAL_DEBUG, combine="max", **TUNE)
    npimg = np.array(image.convert("RGB")); H, W = npimg.shape[:2]
    n = len(per_q) + 1; cols = 3; rows = (n + cols - 1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows)); axes = np.array(axes).reshape(-1)
    for i, (qid, (q, ans, heat, peak)) in enumerate(per_q.items()):
        hu = np.array(PILImage.fromarray((heat*255).astype(np.uint8)).resize((W,H),PILImage.BILINEAR))/255.0
        ax = axes[i]; ax.imshow(npimg); ax.imshow(hu, cmap="jet", alpha=0.45); ax.axis("off")
        ax.set_title(f"q{qid}: {q[:20]}\nA:{ans[:16]!r} pk={peak:.0f}", fontsize=8)
    ax = axes[len(per_q)]
    hu = np.array(PILImage.fromarray((combined*255).astype(np.uint8)).resize((W,H),PILImage.BILINEAR))/255.0
    ax.imshow(npimg); ax.imshow(hu, cmap="jet", alpha=0.55); ax.axis("off")
    ax.set_title(f"★ COMBINED (max)\nqids: {list(CRITICAL_DEBUG.keys())}", fontsize=9, fontweight="bold")
    for j in range(len(per_q)+1, len(axes)): axes[j].axis("off")
    plt.suptitle(f"importance_map — {title}  (SEED={SEED})", fontsize=13, y=1.005)
    plt.tight_layout(rect=[0,0,1,0.99]); plt.show()

# run on a few random cars
for p in rng.sample(panel_records, min(3, len(panel_records))):
    show_importance(p["img"], title=f"panel {p['panel_id']} | GT {p['meta']['plate']}")

In [ ]:
# Algorithm 1, line 9: token-grid map → discrete SCI levels (binary: 2 = critical, 1 = normal)
import numpy as np

TAU = 0.5

def attention_to_sci(att, tau=TAU, levels=None):
    if levels is None:
        return np.where(att > tau, 2, 1).astype(int)
    return (np.digitize(att, levels) + 1).astype(int)

print("attention_to_sci ready")

In [ ]:
# Algorithm 1 (full): image → transmit tensor T = (R, G, B, S, pos)
import numpy as np
from PIL import Image as PILImage

# default critical set: CRITICAL from the importance_map cell (the plate task)

def BuildSCI(image, critical=CRITICAL, tau=TAU, levels=None, combine="max", **tune):
    """
    image    : PIL RGB image (full resolution)
    critical : {qid: score} critical questions (qids index ACTIVE_QUESTION_TYPES)
    tau      : binary threshold (used when levels is None)
    levels   : ascending vector for graded SCI (else binary)
    Returns
      T       : int64 (H, W, 5) = (R, G, B, S, pos)
      A_up    : (H, W) continuous importance map, NEAREST-upsampled   (debug)
      per_q   : {qid: (question, answer, heat, peak)}                 (debug)
      sci_tok : (Hm, Wm) token-grid SCI levels before upsample        (debug)
    """
    npimg = np.array(image.convert("RGB")); H, W = npimg.shape[:2]

    # 1-2. combine (token grid) then threshold (token grid)
    A, per_q = importance_map(image, critical, combine=combine, **tune)    # Hm×Wm, [0,1]
    sci_tok  = attention_to_sci(A, tau=tau, levels=levels)                 # Hm×Wm, {1,2,...}

    # 3. NEAREST-upsample the DISCRETE mask to pixels  (tight, block-aligned)
    S = np.array(PILImage.fromarray(sci_tok.astype(np.uint8))
                 .resize((W, H), PILImage.NEAREST)).astype(np.int64)

    # 4. pack per-pixel tensor T = (R, G, B, S, pos), pos = row*W + col
    rows, cols = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
    pos = (rows * W + cols).astype(np.int64)
    T = np.zeros((H, W, 5), dtype=np.int64)
    T[..., 0:3] = npimg
    T[..., 3]   = S
    T[..., 4]   = pos

    # extra: continuous map upsampled the same way (for debug overlay)
    A_up = np.array(PILImage.fromarray((A*255).astype(np.uint8))
                    .resize((W, H), PILImage.NEAREST)) / 255.0
    return T, A_up, per_q, sci_tok

print("BuildSCI ready (returns T = (R,G,B,S,pos))")

In [ ]:
import random
from matplotlib.colors import ListedColormap, BoundaryNorm
SEED = random.randrange(2**32)
rng = random.Random(SEED)
print(f"[BuildSCI pipeline debug] SEED = {SEED}")

# critical questions here (qid: score). Default = the dataset's critical set.
CRITICAL_SET = dict(CRITICAL)

TUNE = dict(smooth=1.2, dilate=3, sink_drop=4, token_pool="max")

# ── print the critical questions in full (text + score) ──
print(f"\nCritical questions — {len(CRITICAL_SET)} total:")
for qid, score in CRITICAL_SET.items():
    print(f"  q{qid}  (score {score}):  {ACTIVE_QUESTION_TYPES[qid]}")
print()

# pick a few random cars
picks = rng.sample(panel_records, min(3, len(panel_records)))
imgs = [(f"panel {p['panel_id']} | {p['meta']['plate']}", p["img"]) for p in picks]

sci_cmap = ListedColormap(["#3b4cc0", "#d62728"])
norm = BoundaryNorm([0.5,1.5,2.5], sci_cmap.N)

rows = len(imgs)
fig, axes = plt.subplots(rows, 3, figsize=(19, 5.2*rows)); axes = np.atleast_2d(axes)

for r, (label, image) in enumerate(imgs):
    # single BuildSCI call gives everything: tensor T + debug maps
    T, A_up, per_q, sci_tok = BuildSCI(image, CRITICAL_SET, tau=TAU, **TUNE)
    npimg = np.array(image.convert("RGB")); H, W = npimg.shape[:2]
    S = T[..., 3]
    crit = (S == 2).mean()

    axes[r,0].imshow(npimg); axes[r,0].set_title(f"{label} — image"); axes[r,0].axis("off")
    axes[r,1].imshow(npimg); axes[r,1].imshow(A_up, cmap="jet", alpha=0.5)
    axes[r,1].set_title("ATTENTION channel (continuous 0-1)"); axes[r,1].axis("off")
    axes[r,2].imshow(npimg); im=axes[r,2].imshow(S, cmap=sci_cmap, norm=norm, alpha=0.5)
    axes[r,2].set_title(f"SCI channel (τ={TAU}) — {crit:.1%} critical"); axes[r,2].axis("off")
    cbar = fig.colorbar(im, ax=axes[r,2], ticks=[1,2], fraction=0.046, pad=0.04)
    cbar.ax.set_yticklabels(["1 normal","2 critical"])

q_lines = "  |  ".join(f"q{qid}(s{score}): {ACTIVE_QUESTION_TYPES[qid][:22]}"
                       for qid, score in CRITICAL_SET.items())
plt.suptitle(f"BuildSCI pipeline — critical set: {q_lines}  (SEED={SEED})", fontsize=12, y=1.005)
plt.tight_layout(rect=[0,0,1,0.99]); plt.show()

## 4. Tiered transmission over 5G NR OFDM (Sec. III-B)

Sionna link: 30 kHz SCS, 14 OFDM symbols, 128 subcarriers, 3.5 GHz, 3GPP TDL-A (100 ns), perfect CSI,
LMMSE equalization, fixed $E_s/N_0$ for every modulation order.

* **Base layer**: whole image, bilinear downsampling by $D$, uncoded $2^{m_b}$-QAM, upsampled at the receiver.
* **Protection tier**: one self-addressed unit `[pos | R | G | B]` per task-causal pixel, terminated rate-1/3
  convolutional code ($\kappa=7$), BPSK, soft-input Viterbi. Decoded pixels overwrite the base layer.

In [ ]:
import numpy as np, torch, gc
from PIL import Image as PILImage
from sionna.phy.mapping import Mapper, Demapper
from sionna.phy.channel import GenerateOFDMChannel, ApplyOFDMChannel
from sionna.phy.channel.tr38901 import TDL
from sionna.phy.ofdm import ResourceGrid, ResourceGridMapper, LMMSEEqualizer
from sionna.phy.mimo import StreamManagement
from sionna.phy.utils import hard_decisions
from sionna.phy.fec.conv import ConvEncoder, ViterbiDecoder

_dev = "cuda" if torch.cuda.is_available() else "cpu"

# ── OFDM / channel config ──
FFT_SIZE=128; NUM_OFDM_SYM=14; SUBCARRIER_SPACING=30e3
NUM_TX=1; NUM_STREAMS=1
TDL_MODEL="A"; DELAY_SPREAD=100e-9; CARRIER_FREQ=3.5e9; MIN_SPEED=0.0; MAX_SPEED=0.0

RG=ResourceGrid(num_ofdm_symbols=NUM_OFDM_SYM,fft_size=FFT_SIZE,
                subcarrier_spacing=SUBCARRIER_SPACING,num_tx=NUM_TX,
                num_streams_per_tx=NUM_STREAMS,cyclic_prefix_length=6,
                pilot_pattern=None,num_guard_carriers=(0,0),dc_null=False)
DATA_RE_PER_GRID=RG.num_data_symbols
SM=StreamManagement(np.array([[1]]),1)

_tdl=TDL(model=TDL_MODEL,delay_spread=DELAY_SPREAD,carrier_frequency=CARRIER_FREQ,
         min_speed=MIN_SPEED,max_speed=MAX_SPEED)
gen_chan=GenerateOFDMChannel(_tdl,RG,normalize_channel=True)
apply_chan=ApplyOFDMChannel()
rg_mapper=ResourceGridMapper(RG)
lmmse_equ=LMMSEEqualizer(RG,SM)

# ── mapper cache: m=1 → BPSK (pam), even m → QAM ──
_MC={}
def mapper_for(m):
    if m not in _MC:
        if m==1:
            _MC[m]=(Mapper(constellation_type='pam',num_bits_per_symbol=1),
                    Demapper('app',constellation_type='pam',num_bits_per_symbol=1))
        else:
            _MC[m]=(Mapper('qam',m),Demapper('app','qam',m))
    return _MC[m]

# ── conv code for criticals (rate 1/3, BPSK) ──
CONV_RATE=1/3; CONV_CL=7
conv_enc=ConvEncoder(rate=CONV_RATE,constraint_length=CONV_CL,terminate=True)
conv_dec=ViterbiDecoder(encoder=conv_enc,method="soft_llr")

# ── config knobs ──
M_BASE=8; M_CRIT=1; DS=4          # 256-QAM base, BPSK criticals, ↓4× base

def _esn0_no(esn0_db):
    return torch.tensor(10**(-esn0_db/10.0),dtype=torch.float32,device=_dev)
def _bits_msb(v,w):
    v=np.asarray(v,np.int64); s=np.arange(w-1,-1,-1,np.int64)
    return ((v[:,None]>>s)&1).astype(np.uint8)
def _bits_to_int_rows(b):
    w=b.shape[1]; return (b.astype(np.int64)*(1<<np.arange(w-1,-1,-1))).sum(1)

def send_bits_esn0(bits_flat, m, esn0_db):
    """Uncoded bits at QAM order m over genie channel, fixed Es/N0. Returns recovered bits."""
    mp,dm=mapper_for(m); C=DATA_RE_PER_GRID; bpg=C*m
    ng=int(np.ceil(bits_flat.size/bpg)); pad=ng*bpg-bits_flat.size
    b=torch.tensor(np.concatenate([bits_flat,np.zeros(pad,np.uint8)]),
                   dtype=torch.float32,device=_dev).reshape(ng,NUM_TX,NUM_STREAMS,bpg)
    no=_esn0_no(esn0_db); out=[]
    for s in range(0,ng,512):
        x=mp(b[s:s+512]); xr=rg_mapper(x)
        h=gen_chan(batch_size=x.shape[0]); y=apply_chan(xr,h,no)
        xh,ne=lmmse_equ(y,h,torch.zeros_like(no),no)
        out.append(hard_decisions(dm(xh,ne)).to(torch.uint8).reshape(-1).cpu())
    return torch.cat(out).numpy()[:bits_flat.size]

print("transmission foundations ready")
print(f"  base ↓{DS}× @ {2**M_BASE}QAM | criticals conv r={CONV_RATE:.2f} @ {'BPSK' if M_CRIT==1 else str(2**M_CRIT)+'QAM'}")
print(f"  grid C={DATA_RE_PER_GRID} | Es/N0 metric (fixed no)")

In [ ]:
def _send_base_x(npimg, esn0_db, m_base, ds):
    """Base layer: downsample by ds, uncoded 2^m_base-QAM, upsample back to full resolution."""
    H,W=npimg.shape[:2]; Hd,Wd=H//ds,W//ds
    small=np.array(PILImage.fromarray(npimg).resize((Wd,Hd),PILImage.BILINEAR),np.uint8)
    bits=np.unpackbits(small.reshape(-1))
    bhat=send_bits_esn0(bits,m_base,esn0_db)
    rec_small=np.packbits(bhat)[:small.size].reshape(Hd,Wd,3)
    return np.array(PILImage.fromarray(rec_small).resize((W,H),PILImage.BILINEAR),np.uint8)

def _send_crit_conv(crit_idx, crit_rgb, POS, esn0_db, chunk=20000):
    """Critical layer: [POS|R|G|B] per critical pixel → conv rate-1/3 → BPSK → channel → Viterbi."""
    mp,dm=mapper_for(M_CRIT); Lc=len(crit_idx); Uunit=POS+24; no=_esn0_no(esn0_db)
    conv_dec_local=ViterbiDecoder(encoder=conv_enc, method="soft_llr")   # fresh per call → correct shape
    pos_a=np.empty(Lc,np.int64);R_a=np.empty(Lc,np.int64);G_a=np.empty(Lc,np.int64);B_a=np.empty(Lc,np.int64)
    for s in range(0,Lc,chunk):
        e=min(s+chunk,Lc); idx=crit_idx[s:e]; rgb=crit_rgb[s:e]
        u=np.concatenate([_bits_msb(idx,POS),_bits_msb(rgb[:,0],8),
                          _bits_msb(rgb[:,1],8),_bits_msb(rgb[:,2],8)],1)
        u=torch.tensor(u,dtype=torch.float32,device=_dev)
        c=conv_enc(u); n_coded=c.shape[1]; cflat=c.reshape(-1)
        n_units=e-s; total_coded=n_units*n_coded
        C=DATA_RE_PER_GRID; bpg=C*M_CRIT; ng=int(np.ceil(cflat.numel()/bpg)); pad=ng*bpg-cflat.numel()
        cb=torch.cat([cflat,torch.zeros(pad,device=_dev)]).reshape(ng,NUM_TX,NUM_STREAMS,bpg)
        llr=[]
        for g in range(0,ng,512):
            x=mp(cb[g:g+512]); xr=rg_mapper(x)
            h=gen_chan(batch_size=x.shape[0]); y=apply_chan(xr,h,no)
            xh,ne=lmmse_equ(y,h,torch.zeros_like(no),no); llr.append(dm(xh,ne).reshape(-1))
        llr=torch.cat(llr)[:total_coded].reshape(n_units,n_coded)
        uhat=conv_dec_local(llr).to(torch.uint8).cpu().numpy()[:, :Uunit]
        pos_a[s:e]=_bits_to_int_rows(uhat[:,:POS]);R_a[s:e]=_bits_to_int_rows(uhat[:,POS:POS+8])
        G_a[s:e]=_bits_to_int_rows(uhat[:,POS+8:POS+16]);B_a[s:e]=_bits_to_int_rows(uhat[:,POS+16:POS+24])
        del u,c,cb,llr,uhat; gc.collect(); torch.cuda.empty_cache()
    return pos_a,R_a,G_a,B_a

POS_BITS_MAX = 24   # images have different sizes; one fixed address width (covers 1920x1080)

def _sci_mask_allq(img):
    """Critical tasks (plate) → (T, crit_pix, crit_idx, crit_rgb, POS)."""
    CRITICAL_SET={ACTIVE_QUESTION_TYPES.index(q):2 for q in CRITICAL_QUESTIONS}
    T,_,_,_=BuildSCI(img, CRITICAL_SET, tau=TAU, smooth=1.2,dilate=3,sink_drop=4,token_pool="max")
    crit_pix=(T[...,3]==2)
    crit_idx=np.where(crit_pix.reshape(-1))[0]
    crit_rgb=np.array(img,np.uint8).reshape(-1,3)[crit_idx]
    return T,crit_pix,crit_idx,crit_rgb,POS_BITS_MAX

def _to_uint8(img):
    return np.array(img.convert("RGB"),np.uint8) if hasattr(img,"convert") else np.asarray(img,np.uint8)

def transmit_baseline_x(img, esn0_db, m_base, ds):
    """Conventional scheme: whole image downsampled by ds, uncoded 2^m_base-QAM."""
    return _send_base_x(_to_uint8(img), esn0_db, m_base, ds)

def transmit_sci_x(img, esn0_db, m_base, ds, sci_cache):
    """STAR: base layer + conv-coded BPSK critical layer overwriting the critical pixels.
       sci_cache = (crit_idx, crit_rgb, POS, crit_pix) from _sci_mask_allq."""
    npimg=_to_uint8(img); H,W=npimg.shape[:2]; N=H*W
    crit_idx,crit_rgb,POS,crit_pix=sci_cache
    base_img=_send_base_x(npimg, esn0_db, m_base, ds)
    pos,R,G,B=_send_crit_conv(crit_idx,crit_rgb,POS,esn0_db)
    v=pos<N
    sci=base_img.reshape(-1,3).copy(); sci[pos[v]]=np.stack([R,G,B],1)[v]
    return sci.reshape(H,W,3), crit_pix

# shorthands with the default configuration (M_BASE, DS)
def transmit_baseline(img, esn0_db):
    return transmit_baseline_x(img, esn0_db, M_BASE, DS)

def transmit_sci(img, esn0_db, sci_cache=None):
    if sci_cache is None:
        _,crit_pix,crit_idx,crit_rgb,POS=_sci_mask_allq(PILImage.fromarray(_to_uint8(img)))
        sci_cache=(crit_idx,crit_rgb,POS,crit_pix)
    return transmit_sci_x(img, esn0_db, M_BASE, DS, sci_cache)

print("transmit_baseline_x / transmit_sci_x ready")

Sanity check on one car: at 40 dB the plate pixels must be recovered exactly.

In [ ]:
import random, matplotlib.pyplot as plt
SEED=random.randrange(2**32); rng=random.Random(SEED)
print(f"[verify] SEED={SEED}")

panel=rng.choice(panel_records)
img=panel["img"].convert("RGB"); npimg=np.array(img,np.uint8); H,W=npimg.shape[:2]; N=H*W

def psnr(a,b,mask=None):
    a=a.astype(np.float64);b=b.astype(np.float64)
    if mask is not None:a=a[mask];b=b[mask]
    e=np.mean((a-b)**2);return 99 if e==0 else 10*np.log10(255**2/e)

# build SCI mask ONCE, reuse across Es/N0 (this is the expensive VLM step)
_,crit_pix,crit_idx,crit_rgb,POS=_sci_mask_allq(img)
cache=(crit_idx,crit_rgb,POS,crit_pix)
assert POS == POS_BITS_MAX, f"stale POS: cache={POS}, expected {POS_BITS_MAX}"
print(f"{W}×{H} | plate critical={100*crit_pix.mean():.1f}% | |C|={len(crit_idx):,} | POS={POS} | GT={panel['meta']['plate']}")

# airtime
Hd,Wd=H//DS,W//DS
base_syms=int(np.ceil(Hd*Wd*3*8/M_BASE))
n_coded=(POS+24)/CONV_RATE+(CONV_CL-1)/CONV_RATE
crit_syms=int(len(crit_idx)*n_coded/M_CRIT)
print(f"airtime: base(↓{DS}×)={base_syms:,} sym | crit(conv-BPSK)={crit_syms:,} sym | "
      f"total={base_syms+crit_syms:,} ({(base_syms+crit_syms)/base_syms:.1f}× base-only)\n")

# ── (1) noiseless sanity at 40 dB ──
b40=transmit_baseline(img,40)
s40,_=transmit_sci(img,40,sci_cache=cache)
crit_exact=np.array_equal(s40.reshape(-1,3)[crit_idx], crit_rgb)
print(f"(1) noiseless@40dB: base blurry-PSNR={psnr(npimg,b40):.1f} | "
      f"SCI critical pixels exact={crit_exact} | SCI crit-PSNR={psnr(npimg,s40,crit_pix):.1f}")

# ── SCI-channel overlay ──
from matplotlib.colors import ListedColormap,BoundaryNorm
cm=ListedColormap(["#3b4cc0","#d62728"]); nm=BoundaryNorm([0.5,1.5,2.5],2)
fig,ax=plt.subplots(1,2,figsize=(14,5))
ax[0].imshow(npimg);ax[0].set_title(f"original {W}×{H}");ax[0].axis("off")
ax[1].imshow(npimg);im=ax[1].imshow(crit_pix.astype(int)+1,cmap=cm,norm=nm,alpha=0.45)
ax[1].set_title(f"SCI channel (plate) — {100*crit_pix.mean():.1f}% critical");ax[1].axis("off")
cb=fig.colorbar(im,ax=ax[1],ticks=[1,2],fraction=0.046);cb.ax.set_yticklabels(["normal","critical"])
plt.tight_layout();plt.show()

# ── (2) visual sweep: baseline vs SCI across Es/N0 ──
SHOW=[-8,-4,0,4,8,12,16,20]
rows=[]
for esn0 in SHOW:
    b=transmit_baseline(img,esn0)
    s,_=transmit_sci(img,esn0,sci_cache=cache)
    rows.append(dict(esn0=esn0,base=b,sci=s,
                     base_c=psnr(npimg,b,crit_pix),sci_c=psnr(npimg,s,crit_pix)))
    print(f"  Es/N0={esn0:3d} | SCI crit={rows[-1]['sci_c']:.1f} | BASE crit={rows[-1]['base_c']:.1f}")

fig,axes=plt.subplots(3,len(rows),figsize=(5.5*len(rows),12)); axes=np.atleast_2d(axes)
for j,r in enumerate(rows):
    axes[0,j].imshow(npimg);axes[0,j].axis("off");axes[0,j].set_title("original")
    axes[1,j].imshow(r["sci"]);axes[1,j].axis("off");axes[1,j].set_title(f"SCI {r['esn0']}dB\ncrit={r['sci_c']:.1f} dB")
    axes[2,j].imshow(r["base"]);axes[2,j].axis("off");axes[2,j].set_title(f"BASE {r['esn0']}dB\ncrit={r['base_c']:.1f} dB")
plt.suptitle(f"Verify: SCI (plate, conv-BPSK crit) vs downsampled baseline (SEED={SEED})",y=1.005,fontsize=13)
plt.tight_layout(rect=[0,0,1,0.99]); plt.show()

## 5. Semantic evaluation: speaker VLM + LLM judge

The **speaker** (Qwen3-VL-30B-A3B, served by Ollama) answers each query on the received image, and the
**judge** (Qwen3-8B, Ollama) decides whether the answer is semantically equivalent to the ground truth
(S-Acc, Sec. IV-A). S-PSNR (Eq. 6) is the PSNR over the ground-truth task-causal region only.

```bash
ollama pull qwen3-vl:30b-a3b-instruct
ollama pull qwen3:8b
```

In [ ]:
import requests, re, io, base64
import numpy as np
from PIL import Image as PILImage

OLLAMA_NUM_CTX = 8192   # OLLAMA_URL, SPEAKER_MODEL, JUDGE_MODEL come from the model-manager cell

def _img_to_b64(img):
    """PIL or uint8 array → base64 PNG (matches encode_image convention)."""
    if not hasattr(img,"save"): img=PILImage.fromarray(np.asarray(img,np.uint8))
    buf=io.BytesIO(); img.save(buf,format="PNG"); return base64.b64encode(buf.getvalue()).decode("utf-8")

def ask_vision(prompt, img, model=None, timeout=300, max_tokens=256):
    """Single-image VLM query via Ollama."""
    payload={"model":model or SPEAKER_MODEL,"prompt":prompt,"images":[_img_to_b64(img)],
             "stream":False,"keep_alive":"30m","think":False,
             "options":{"temperature":0,"top_p":1,"top_k":1,"num_ctx":OLLAMA_NUM_CTX,"num_predict":max_tokens}}
    r=requests.post(OLLAMA_URL,json=payload,timeout=timeout); d=r.json()
    if "error" in d: raise RuntimeError(d["error"])
    return d.get("response","").strip()

def ask_text(prompt, model=None, timeout=120, max_tokens=64):
    """Text-only query via Ollama (judge)."""
    payload={"model":model or JUDGE_MODEL,"prompt":prompt,"stream":False,"keep_alive":"30m","think":False,
             "options":{"temperature":0,"top_p":1,"top_k":1,"num_ctx":OLLAMA_NUM_CTX,"num_predict":max_tokens}}
    r=requests.post(OLLAMA_URL,json=payload,timeout=timeout); d=r.json()
    if "error" in d: raise RuntimeError(d["error"])
    return d.get("response","").strip()

def parse_answer_text(reply):
    """Extract <answer>...</answer>, else fall back to the raw reply."""
    m=re.search(r"<answer>(.*?)</answer>", reply, re.DOTALL|re.IGNORECASE)
    return (m.group(1) if m else reply).strip()

def truth_to_text(truth):
    if isinstance(truth,bool): return "yes" if truth else "no"
    return str(truth)

def full_image_question_answer(img, question):
    """Ask the VLM `question` about `img`. Returns (parsed_answer, raw_reply)."""
    prompt=f"""Answer the question using only the image.

QUESTION:
{question}

Rules:
- Do not explain.
- Return only the final answer.

Return only in this format:
<answer>final answer here</answer>
"""
    reply=ask_vision(prompt, img)
    return parse_answer_text(reply), reply

def llm_judge_equivalent(question, truth, model_answer):
    """Judge semantic equivalence of model_answer vs ground-truth. Returns (bool, verdict)."""
    truth_text=truth_to_text(truth)
    prompt=f"""You are grading a semantic communication result.

Decide whether the MODEL ANSWER means the same as the CORRECT ANSWER for the given question.

QUESTION:
{question}

CORRECT ANSWER:
{truth_text}

MODEL ANSWER:
{model_answer}

Rules:
- Judge only semantic equivalence.
- Ignore XML tags, capitalization, and extra words if meaning is preserved.
Reply with only one word: YES or NO
"""
    verdict=ask_text(prompt, model=JUDGE_MODEL)
    return verdict.strip().upper().startswith("Y"), verdict

# ── quick connectivity + sanity test on ONE clean image ──
_p=panel_records[0]
_ans,_raw=full_image_question_answer(_p["img"].convert("RGB"), ACTIVE_QUESTION_TYPES[0])
_gt=answer_for_question(_p["meta"], ACTIVE_QUESTION_TYPES[0])
_ok,_v=llm_judge_equivalent(ACTIVE_QUESTION_TYPES[0], _gt, _ans)
print("Ollama eval infra ready.")
print(f"  Q: {ACTIVE_QUESTION_TYPES[0]}")
print(f"  VLM answer: {_ans!r} | ground truth: {_gt!r} | judge: {_v} → correct={_ok}")

## 6. Plate experiment (Fig. 8 and Fig. 9)

Configurations: original image (no channel, upper bound), 256-QAM at $D=4$, 256-QAM at $D=1$, and STAR
(256-QAM base, $D=4$). S-PSNR is computed while the reconstructions are in memory, then speaker and judge run
one model at a time.

In [ ]:
import random, os, pickle, time, gc, numpy as np, torch, matplotlib.pyplot as plt
from PIL import Image as PILImage

SAVE_DIR=RESULTS_DIR; print("SAVE_DIR:",SAVE_DIR)

QPLATE = ACTIVE_QUESTION_TYPES[0]
QSET   = [QPLATE]
M_BASE_EXP=8; ESN0_SWEEP=[-8,-6,-4,-2,0,1,2,4,8,12,16]
SEED_RUN=20240612
POS_BITS=POS_BITS_MAX; RATE=CONV_RATE; CL=CONV_CL

CONFIGS={"Original":"#111827",                 # no-channel upper bound
         "256-QAM DS=4":"#DC2626",             # conv, compressed
         "256-QAM DS=1":"#2563EB",             # conv, uncompressed
         "STAR (256-QAM, DS=4)":"#F5C518"}     # STAR — yellow
CHAN_CFGS=[c for c in CONFIGS if c!="Original"]   # the 3 transmitted configs

N_IMG = min(len(panel_records),N_PANELS  )
rng=random.Random(SEED_RUN)
test_imgs=rng.sample(panel_records, N_IMG)
print(f"{N_IMG} imgs | question: {QPLATE!r}\n")

def _spsnr(recon, mk):
    """PSNR over the semantically critical (plate) pixels only."""
    m=mk["crit_pix"].reshape(-1)
    a=mk["npimg"].astype(np.float64).reshape(-1,3)[m]
    b=recon.astype(np.float64).reshape(-1,3)[m]
    e=np.mean((a-b)**2)
    return 99.0 if e==0 else float(10*np.log10(255**2/e))

# ══ PHASE 0: HF attention → plate SCI mask per image → free HF ══
print("[Phase 0] HF Qwen2.5-VL → plate SCI masks")
load_hf_model()
masks=[]
for panel in test_imgs:
    img=panel["img"].convert("RGB"); npimg=np.array(img,np.uint8)
    _,crit_pix,crit_idx,crit_rgb,POS=_sci_mask_allq(img)
    masks.append(dict(npimg=npimg, plate=panel["meta"]["plate"],
                      cache=(crit_idx,crit_rgb,POS,crit_pix),
                      crit_pix=crit_pix, n_crit=len(crit_idx)))
    print(f"  {panel['meta']['plate']:10s} {npimg.shape[1]}×{npimg.shape[0]} | crit={100*crit_pix.mean():.2f}% | |C|={len(crit_idx):,}", flush=True)
free_hf_model()

# ══ PHASE 1: transmit → store recons + S-PSNR (computed while recon exists) ══
print("\n[Phase 1] transmission + S-PSNR (no LLM)")
recons={}; spsnr={}
for ii,(panel,mk) in enumerate(zip(test_imgs,masks)):
    img=panel["img"].convert("RGB")
    recons[(ii,None,"Original")] = mk["npimg"]
    for e in ESN0_SWEEP:
        r_d4=transmit_baseline_x(img,e,M_BASE_EXP,4)
        r_d1=transmit_baseline_x(img,e,M_BASE_EXP,1)
        r_st=transmit_sci_x(img,e,M_BASE_EXP,4,mk["cache"])[0]
        recons[(ii,e,"256-QAM DS=4")]=r_d4
        recons[(ii,e,"256-QAM DS=1")]=r_d1
        recons[(ii,e,"STAR (256-QAM, DS=4)")]=r_st
        spsnr[(ii,e,"256-QAM DS=4")]=_spsnr(r_d4,mk)
        spsnr[(ii,e,"256-QAM DS=1")]=_spsnr(r_d1,mk)
        spsnr[(ii,e,"STAR (256-QAM, DS=4)")]=_spsnr(r_st,mk)
    gc.collect(); torch.cuda.empty_cache()
    print(f"  transmit img {ii+1}/{N_IMG} ({mk['plate']})", flush=True)
print(f"  {len(recons)} reconstructions stored")

# ══ PHASE 2: speaker VLM → answers  (per-image progress) ══
print("\n[Phase 2] speaker VLM → answers")
load_ollama(SPEAKER_MODEL)
answers={}
n_cfg_per_img = 1 + len(CHAN_CFGS)*len(ESN0_SWEEP)     # answers expected per image
done=0
for (ii,e,cfg),recon in recons.items():
    try: a,_=full_image_question_answer(PILImage.fromarray(recon),QPLATE)
    except Exception as ex: a=f"(fail:{str(ex)[:15]})"
    answers[(ii,e,cfg)]=a
    done+=1
    if done % n_cfg_per_img == 0:                       # one line per image finished
        print(f"  speaker img {done//n_cfg_per_img}/{N_IMG} done ({done} answers)", flush=True)
print(f"  {len(answers)} answers total")
unload_ollama(SPEAKER_MODEL)
del recons; gc.collect(); torch.cuda.empty_cache()

# ══ PHASE 3: judge → grade ══
print("\n[Phase 3] judge → grade")
load_ollama(JUDGE_MODEL)
records=[]
for ii,(panel,mk) in enumerate(zip(test_imgs,masks)):
    gt=answer_for_question(panel["meta"],QPLATE)
    for cfg in CONFIGS:
        snr_list=[None] if cfg=="Original" else ESN0_SWEEP
        for e in snr_list:
            ans=answers[(ii,e,cfg)]
            try: ok,_=llm_judge_equivalent(QPLATE,gt,ans)
            except Exception as ex: print("  judge fail",str(ex)[:30]); ok=False
            records.append(dict(config=cfg,img_idx=ii,plate=mk["plate"],snr=e,
                                answer=ans,truth=gt,correct=bool(ok)))
    print(f"  judged img {ii+1}/{N_IMG}", flush=True)
unload_ollama(JUDGE_MODEL)

# ══ PHASE 4: aggregate + RE + save + plots ══
print("\n[Phase 4] aggregate + save")
def acc(cfg,e):
    v=[r["correct"] for r in records if r["config"]==cfg and r["snr"]==e]
    return float(np.mean(v)) if v else 0.0
orig_acc=acc("Original",None)
acc_curve={c:{e:acc(c,e) for e in ESN0_SWEEP} for c in CHAN_CFGS}

def spsnr_mean(cfg,e):
    v=[spsnr[(ii,e,cfg)] for ii in range(N_IMG) if (ii,e,cfg) in spsnr]
    return float(np.mean(v)) if v else float("nan")
spsnr_curve={c:{e:spsnr_mean(c,e) for e in ESN0_SWEEP} for c in CHAN_CFGS}

def base_re(px,m): return int(np.ceil(px*24/m))
def crit_re(n): u=POS_BITS+24; return int(np.ceil(n*(u/RATE+(CL-1)/RATE)))
re_vals={c:[] for c in CHAN_CFGS}
for mk in masks:
    N=mk["npimg"].shape[0]*mk["npimg"].shape[1]
    re_vals["256-QAM DS=4"].append(base_re(N//16,M_BASE_EXP))
    re_vals["256-QAM DS=1"].append(base_re(N,M_BASE_EXP))
    re_vals["STAR (256-QAM, DS=4)"].append(base_re(N//16,M_BASE_EXP)+crit_re(mk["n_crit"]))
rm={k:float(np.mean(v)) for k,v in re_vals.items()}

stamp=time.strftime("%Y%m%d_%H%M%S")
fname=os.path.join(SAVE_DIR,f"plate_experiment_{stamp}.pkl")
with open(fname,"wb") as f:
    pickle.dump({"records":records,"acc_curve":acc_curve,"orig_acc":orig_acc,
                 "spsnr_curve":spsnr_curve,
                 "re_vals":re_vals,"re_mean":rm,"QPLATE":QPLATE,"ESN0_SWEEP":ESN0_SWEEP,
                 "CONFIGS":list(CONFIGS.keys()),
                 "config":{"M_BASE":M_BASE_EXP,"RATE":RATE,"CL":CL,"POS_BITS":POS_BITS,
                           "SEED_RUN":SEED_RUN,"N_IMAGES":N_IMG},
                 "plates":[m["plate"] for m in masks]},f)
print(f"★ saved {fname} ({len(records)} records)")

# ── accuracy curve ──
fig,ax=plt.subplots(figsize=(8,5))
for cfg in acc_curve:
    ax.plot(ESN0_SWEEP,[acc_curve[cfg][e] for e in ESN0_SWEEP],"o-",color=CONFIGS[cfg],lw=2.2,label=cfg)
ax.axhline(orig_acc,ls="--",color=CONFIGS["Original"],lw=1.5,label=f"Original (no channel) = {orig_acc:.2f}")
ax.set_xlabel("Es/N0 (dB)");ax.set_ylabel("plate accuracy");ax.set_ylim(0,1.03)
ax.grid(alpha=0.3);ax.legend(fontsize=9,loc="lower right");ax.set_title("Plate accuracy vs Es/N0")
plt.tight_layout(); plt.show()

# ── S-PSNR curve ──
fig,ax=plt.subplots(figsize=(8,5))
for cfg in spsnr_curve:
    ax.plot(ESN0_SWEEP,[spsnr_curve[cfg][e] for e in ESN0_SWEEP],"o-",color=CONFIGS[cfg],lw=2.2,label=cfg)
ax.set_xlabel("Es/N0 (dB)");ax.set_ylabel("S-PSNR (dB)")
ax.grid(alpha=0.3);ax.legend(fontsize=9,loc="lower right")
ax.set_title("Semantically Critical Region PSNR (S-PSNR) vs Es/N0")
plt.tight_layout(); plt.show()

# ── RE bar ──
fig,axb=plt.subplots(figsize=(7,5))
keys=list(re_vals.keys())
bars=axb.bar(range(len(keys)),[rm[k]/1e6 for k in keys],color=[CONFIGS[k] for k in keys],edgecolor="black")
axb.set_xticks(range(len(keys)));axb.set_xticklabels(keys,fontsize=8,rotation=12)
axb.set_ylabel("Resource elements per image (millions)");axb.set_title("Resources used");axb.grid(alpha=0.3,axis="y")
for b,k in zip(bars,keys): axb.text(b.get_x()+b.get_width()/2,rm[k]/1e6*1.02,f"{rm[k]/1e6:.2f}M",ha="center",fontsize=8,fontweight="bold")
plt.tight_layout(); plt.show()
for k in keys: print(f"{k:24s}: {rm[k]/1e6:.3f}M REs")

Paper-style figures: plate S-Acc (Fig. 8a), S-PSNR (Fig. 8b) and REs per image (Fig. 9).

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt

SAVE_DIR = FIG_DIR

INK="#000000"; GRID_COLOR="#D8DEE9"
plt.rcParams.update({
    "font.family":"sans-serif",
    "font.sans-serif":["TeX Gyre Heros","Liberation Sans","Helvetica","Arial","DejaVu Sans"],
    "figure.dpi":180,"figure.facecolor":"white","axes.facecolor":"white",
    "text.color":INK,"axes.labelcolor":INK,"xtick.color":INK,"ytick.color":INK,
    "xtick.labelsize":8.5,"ytick.labelsize":8.5,"axes.labelsize":9.5,
    "pdf.fonttype":42,"ps.fonttype":42,
})
def style_axis(ax):
    ax.grid(True,linestyle="--",color=GRID_COLOR,linewidth=0.7,alpha=0.85); ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    for s in ("left","bottom"): ax.spines[s].set_color(INK); ax.spines[s].set_linewidth(1.0)
    ax.tick_params(axis="both",length=4,width=1.0,color=INK)

CHAN=[c for c in CONFIGS if c!="Original"]           # DS=4, DS=1, STAR (colors from CONFIGS)

# ── (1) plate accuracy ─────────────────────────────────────────────────────
fig,ax=plt.subplots(figsize=(3.5,2.7))
for cfg in CHAN:
    ax.plot(ESN0_SWEEP,[acc_curve[cfg][e] for e in ESN0_SWEEP],"o-",color=CONFIGS[cfg],lw=1.6,ms=3.5,label=cfg)
ax.axhline(orig_acc,ls="--",color=CONFIGS["Original"],lw=1.2,label=f"Original = {orig_acc:.2f}")
ax.set_xlabel(r"$E_s/N_0$ (dB)"); ax.set_ylabel("Semantic accuracy"); ax.set_ylim(0,1.03)
style_axis(ax)
leg=ax.legend(fontsize=6.0, loc="upper left", framealpha=0.9, handlelength=1.6,
              handletextpad=0.4, labelspacing=0.25, borderpad=0.35)
leg.get_frame().set_edgecolor(GRID_COLOR); leg.get_frame().set_linewidth(0.6)
fig.tight_layout(pad=0.6)
out=os.path.join(SAVE_DIR,"plate_accuracy_vs_EsN0.pdf"); fig.savefig(out,format="pdf",bbox_inches="tight")
print("Saved:",out); plt.show(); plt.close(fig)

# ── (2) S-PSNR ─────────────────────────────────────────────────────────────
fig,ax=plt.subplots(figsize=(3.5,2.7))
for cfg in CHAN:
    ax.plot(ESN0_SWEEP,[spsnr_curve[cfg][e] for e in ESN0_SWEEP],"o-",color=CONFIGS[cfg],lw=1.6,ms=3.5,label=cfg)
ax.set_xlabel(r"$E_s/N_0$ (dB)"); ax.set_ylabel("S-PSNR (dB)")
style_axis(ax)
leg=ax.legend(fontsize=6.0,loc="upper left",framealpha=0.9,handlelength=1.6,
              handletextpad=0.4,labelspacing=0.25,borderpad=0.35)
leg.get_frame().set_edgecolor(GRID_COLOR); leg.get_frame().set_linewidth(0.6)
fig.tight_layout(pad=0.6)
out=os.path.join(SAVE_DIR,"plate_S-PSNR_vs_EsN0.pdf"); fig.savefig(out,format="pdf",bbox_inches="tight")
print("Saved:",out); plt.show(); plt.close(fig)

# ── (3) resource-saving bar (POS=24, matched conventional) ─────────────────
POS_BITS=POS_BITS_MAX; RATE=CONV_RATE; CL=CONV_CL; BASE_QAM=8; STAR_DS=4
def _base_re(px,m): return int(np.ceil(px*24/m))
def _crit_re(n): u=POS_BITS+24; return int(np.ceil(n*(u/RATE+(CL-1)/RATE)))
bpsk_conv=[]; star=[]
for mk in masks:
    N=mk["npimg"].shape[0]*mk["npimg"].shape[1]; nc=int(mk["crit_pix"].sum())
    bpsk_conv.append(int(np.ceil(N*24/RATE)))
    star.append(_base_re(N//(STAR_DS*STAR_DS),BASE_QAM)+_crit_re(nc))
m_bc=np.mean(bpsk_conv); m_st=np.mean(star); saved=m_bc-m_st
vals=[m_bc,m_st,saved]; labels=["BPSK +\nConv-1/3","STAR\n(256-QAM)","REs\nsaved"]
colors=["#6B7280",CONFIGS["STAR (256-QAM, DS=4)"],"#16A34A"]
fig,ax=plt.subplots(figsize=(3.45,2.5))
bars=ax.bar(range(3),np.array(vals)/1e6,color=colors,width=0.62,edgecolor=INK,linewidth=0.8,zorder=3)
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_ylabel("Resource elements per image\n(in Millions)",linespacing=1.2)
ax.set_ylim(0,max(vals)/1e6*1.16); style_axis(ax); ax.grid(axis="x",visible=False)
for b,v in zip(bars,vals):
    ax.text(b.get_x()+b.get_width()/2,v/1e6+max(vals)/1e6*0.02,f"{v/1e6:.1f}M",
            ha="center",va="bottom",fontsize=8.6,fontweight="bold",color=INK,clip_on=False)
fig.tight_layout(pad=0.7); fig.subplots_adjust(left=0.17)
out=os.path.join(SAVE_DIR,"plate_resource_saving_BPSKconv_vs_STAR256_DS4.pdf")
fig.savefig(out,format="pdf",bbox_inches="tight")
print("Saved:",out); plt.show(); plt.close(fig)
print(f"\nBPSK+Conv-1/3: {m_bc/1e6:.2f}M | STAR: {m_st/1e6:.2f}M | saved: {saved/1e6:.2f}M ({100*saved/m_bc:.0f}%)")

## 7. Received images (Fig. 7)

One car at $E_s/N_0 = 10$ dB: original, 256-QAM at $D=4$ and $D=1$, STAR, and the SCI map of the plate task.
Set `REDACT = True` to blur the plate in the saved images.

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from PIL import Image as PILImage, ImageFilter
from scipy.ndimage import binary_dilation

SAVE_DIR = FIG_DIR
load_hf_model()

IMG_PATH  = os.path.join(DATA_DIR, "us1.jpg")
ESN0_SHOW = 10
REDACT    = False

plt.rcParams.update({
    "font.family":"sans-serif",
    "font.sans-serif":["TeX Gyre Heros","Liberation Sans","Helvetica","Arial","DejaVu Sans"],
    "figure.dpi":180,"figure.facecolor":"white","pdf.fonttype":42,"ps.fonttype":42,
})

# ground-truth plate from the paired .txt (optional, just for the print)
_txt = os.path.splitext(IMG_PATH)[0] + ".txt"
gt = open(_txt).read().strip().split("\t")[-1].strip().upper() if os.path.exists(_txt) else "?"

img = PILImage.open(IMG_PATH).convert("RGB")
npimg = np.array(img, np.uint8); H,W = npimg.shape[:2]

# build the plate SCI mask for THIS image (needs HF model loaded)
_,crit_pix,crit_idx,crit_rgb,POS = _sci_mask_allq(img)
cache = (crit_idx, crit_rgb, POS, crit_pix)
print(f"{os.path.basename(IMG_PATH)} | GT={gt} | {W}x{H} | crit={100*crit_pix.mean():.2f}% | |C|={len(crit_idx):,}")

def _redact(arr):
    if not REDACT: return arr
    blurred = np.array(PILImage.fromarray(arr.astype(np.uint8))
                       .filter(ImageFilter.GaussianBlur(radius=max(H,W)//80)))
    m = binary_dilation(crit_pix, iterations=6)
    out = arr.copy(); out[m] = blurred[m]; return out

def save_img(arr, suffix, overlay=None):
    shown = _redact(arr)
    fig_w=2.2; fig_h=fig_w*H/W
    fig,ax=plt.subplots(figsize=(fig_w,fig_h))
    ax.imshow(shown)
    if overlay is not None: ax.imshow(overlay, cmap="jet", alpha=0.45)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(True); s.set_color("#000000"); s.set_linewidth(0.8)
    ax.margins(0); fig.subplots_adjust(left=0,right=1,top=1,bottom=0)
    out=os.path.join(SAVE_DIR,f"plate_panel_{ESN0_SHOW}dB_{suffix}.pdf")
    fig.savefig(out,format="pdf",bbox_inches="tight",pad_inches=0.02)
    print("Saved:",out); plt.show(); plt.close(fig)

d4     = transmit_baseline_x(img, ESN0_SHOW, 8, 4)
d1     = transmit_baseline_x(img, ESN0_SHOW, 8, 1)
star,_ = transmit_sci_x(img, ESN0_SHOW, 8, 4, cache)
A,_ = importance_map(img, CRITICAL, combine="max", smooth=1.2, dilate=3, sink_drop=4, token_pool="max")
A_up = np.array(PILImage.fromarray((A*255).astype(np.uint8)).resize((W,H), PILImage.BILINEAR))/255.0

save_img(npimg, "original")
save_img(d4,    "conv_256QAM_DS4")
save_img(d1,    "conv_256QAM_DS1")
save_img(star,  "STAR_256QAM_DS4")
save_img(npimg, "SCI_overlay", overlay=A_up)
print("Done.")